Librerias

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# 1. Carregar les dades

In [2]:
# 1. Carregar les dades reals directament
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Éxito al cargar datos reales.")
print(f"Dimensiones de entrenamiento: {train_df.shape}")  # Debería mostrar (8693, 14)
print(f"Dimensiones de testeo: {test_df.shape}")         # Debería mostrar (4277, 13)

Éxito al cargar datos reales.
Dimensiones de entrenamiento: (8693, 14)
Dimensiones de testeo: (4277, 13)


# 2. Enginyeria de característiques (Feature Engineering)

In [3]:
def preprocess_features(df):
    df = df.copy()
    
    # 1. Gastos: Imputar nulos con 0 en lugar de eliminarlos (los que no gastan suelen tener NaN)
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df[spend_cols] = df[spend_cols].fillna(0)
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    
    # 2. Nueva feature: Indicador de si el pasajero gastó algo (muy potente para predecir CryoSleep)
    df['HasSpented'] = (df['TotalSpend'] > 0).astype(int)
    
    # 3. Extraer grupo del PassengerId y calcular el tamaño del grupo familiar
    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0] if pd.notnull(x) else '0000')
    group_sizes = df['Group'].value_counts()
    df['GroupSize'] = df['Group'].map(group_sizes)
    
    # 4. Nueva feature: ¿Viaja solo?
    df['IsAlone'] = (df['GroupSize'] == 1).astype(int)
    
    # 5. Extraer detalles de la cabina de manera segura
    df['Cabin'] = df['Cabin'].fillna('U/0/U')
    df['Cabin_Deck'] = df['Cabin'].apply(lambda x: x.split('/')[0])
    df['Cabin_Side'] = df['Cabin'].apply(lambda x: x.split('/')[-1])
    
    return df

train_processed = preprocess_features(train_df)
test_processed = preprocess_features(test_df)

# Definir columnas numéricas y categóricas actualizadas
num_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend', 'GroupSize']
cat_features = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Cabin_Deck', 'Cabin_Side', 'HasSpented', 'IsAlone']

# Pipelines de preprocesamiento
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ])

# Separar características y target
X = train_processed[num_features + cat_features]
y = train_processed['Transported'].astype(int).values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Aplicar transformaciones
X_train_trans = preprocessor.fit_transform(X_train)
X_val_trans = preprocessor.transform(X_val)
print(f"Forma de les dades d'entrenament transformades: {X_train_trans.shape}")

Forma de les dades d'entrenament transformades: (6954, 34)


# 3. Arquitectura de la Xarxa Neuronal (MLP)

In [5]:
# Añado más profundidad con una estructura piramidal para destilar la información de forma progresiva.
model = keras.Sequential([
    keras.layers.Input(shape=(X_train_trans.shape[1],)),
    
    # Capa 1 (Entrada de alta dimensión)
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    
    # Capa 2
    keras.layers.Dense(64, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    
    # Capa 3 (NUEVA: Estructura intermedia para abstraer patrones complejos)
    keras.layers.Dense(32, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    
    # Capa 4 (NUEVA: Destilación final antes de la clasificación)
    keras.layers.Dense(16, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.1),
    
    # Capa de salida
    keras.layers.Dense(1, activation='sigmoid')
])

# Usamos un optimizador Adam con un Learning Rate ligeramente más controlado (0.001 por defecto)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Early stopping modificado: Aumentamos el "patience" a 12 para dar margen de aprendizaje a la red más profunda
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=12,
    restore_best_weights=True
)

# Entrenament del model

In [6]:
history = model.fit(
    X_train_trans, y_train,
    validation_data=(X_val_trans, y_val),
    epochs=60,             # Aumentamos ligeramente el tope ya que la red es más grande
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluación en validación local
val_preds = (model.predict(X_val_trans) > 0.5).astype(int)
acc = accuracy_score(y_val, val_preds)
print(f"\nAccuracy Final de Validació: {acc:.4f}")
print("\nInforme de Classificació:")
print(classification_report(y_val, val_preds))

Epoch 1/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7131 - loss: 0.5655 - val_accuracy: 0.7757 - val_loss: 0.4993
Epoch 2/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7693 - loss: 0.4825 - val_accuracy: 0.7809 - val_loss: 0.4345
Epoch 3/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7794 - loss: 0.4618 - val_accuracy: 0.7861 - val_loss: 0.4280
Epoch 4/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7849 - loss: 0.4511 - val_accuracy: 0.7844 - val_loss: 0.4224
Epoch 5/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7866 - loss: 0.4456 - val_accuracy: 0.7838 - val_loss: 0.4206
Epoch 6/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7915 - loss: 0.4418 - val_accuracy: 0.7815 - val_loss: 0.4265
Epoch 7/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7886 - loss: 0.4353 - val_accuracy: 0.7867 - val_loss: 0.4162
Epoch 8/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7987 - loss: 0.4307 - val_accuracy: 0.

# 4. Generar el archivo de envío para Kaggle

In [7]:
print("\nPreparando los datos de testeo para predicción...")
X_test_real = test_processed[num_features + cat_features]
X_test_trans = preprocessor.transform(X_test_real)

print("Realizando predicciones con el modelo profundo...")
predictions_proba = model.predict(X_test_trans)
predictions_bool = (predictions_proba >= 0.5).astype(bool).flatten()

submission_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': predictions_bool
})

output_filename = 'submission_deep_mlp.csv'
submission_df.to_csv(output_filename, index=False)

print(f"\n¡Proceso completado!")
print(f"Tu nuevo archivo de entrega optimizado: '{output_filename}'")
print(submission_df.head())


Preparando los datos de testeo para predicción...
Realizando predicciones con el modelo profundo...
134/134 ━━━━━━━━━━━━━━━━━━━━ 0s 739us/step

¡Proceso completado!
Tu nuevo archivo de entrega optimizado: 'submission_deep_mlp.csv'
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
